# Laya reproducibility check: PyTorch on a Colab GPU against the benchmark's MLX run

The benchmark flew Laya v4 as an MLX build on an M3 MacBook Pro (16 GB). This notebook loads the same fine-tuned checkpoint in PyTorch on a CUDA GPU and asks it every question in Laya's benchmark recordings, at the same game states (`pnpm agree`). It reports how often both builds pick the same manoeuvre and how far the probabilities move. It also times Laya on the same GPU type as Kev, so the self-hosted lanes' answer times can be compared.

Baseline, measured on the Mac (2026-10-31): the MLX build against its own recordings picked the same manoeuvre on 1,416 of 1,416 decisions (courses 5001–5005). The tiny remaining distance comes from the recordings rounding probabilities to 4 decimals.

**Before you start**
- Runtime → Change runtime type → **G4 GPU** (the same as the Kev lanes).
- The checkpoint comes from the Hub revision recorded in `repro/laya-descent-v4-release.json`, and every file is checked against the hashes there. Until a revision is recorded, put `laya-descent-v4.zip` (about 780 MB, from `bridge/laya/models/` after fine-tuning) into **My Drive/lowgate/** in Google Drive.
- Only for a private fork: a `GITHUB_TOKEN` in Colab Secrets, as in the Kev notebook.
- Runtime → Run all. `laya-repro.zip` downloads at the end: unzip it in the repo root on your computer.

In [ ]:
COURSES = "5001-5020"   # all of Laya's benchmark courses: 5,801 answered decisions
LABEL = "torch-g4-hub"  # names the report: repro/laya-<LABEL>.json
LAYA = "0.4.17"         # the laya version the checkpoint was fine-tuned with (finetune_colab.ipynb)
NODE = "v24.21.0"

In [ ]:
# 1. GPU and Laya (PyTorch is preinstalled on Colab).
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q laya=={LAYA}

In [ ]:
# 2. Node and pnpm, pinned to the versions the local lanes were flown with.
!curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /opt
import os
os.environ["PATH"] = f"/opt/node-{NODE}-linux-x64/bin:" + os.environ["PATH"]
os.environ["COREPACK_ENABLE_DOWNLOAD_PROMPT"] = "0"
!corepack enable && node -v && pnpm -v

In [ ]:
# 3. The Low Gate repo. A GITHUB_TOKEN in Colab Secrets is used only if present (for a private fork); it stays out of the output and out of .git/config.
from google.colab import userdata
import subprocess
REPO = "github.com/ErenTerakye/lowgate.git"
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
if not os.path.exists("/content/lowgate"):
    url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"
    r = subprocess.run(["git", "clone", "-q", url, "/content/lowgate"], capture_output=True, text=True)
    assert r.returncode == 0, "Clone failed: check the repository URL (and GITHUB_TOKEN, if the repository is private)"
    subprocess.run(["git", "-C", "/content/lowgate", "remote", "set-url", "origin", f"https://{REPO}"], check=True)
del token
%cd /content/lowgate
!git log --oneline -1 && pnpm install --frozen-lockfile --silent

In [ ]:
# 4. The checkpoint: from the pinned Hub revision if one is recorded, otherwise from Drive. Either way, every file must match its hash.
import hashlib, json, pathlib, zipfile
manifest = json.load(open("repro/laya-descent-v4-release.json"))
hub = manifest["hub"]
if hub["repo"]:
    from huggingface_hub import snapshot_download
    CHECKPOINT = str(pathlib.Path(snapshot_download(hub["repo"], revision=hub["revision"], allow_patterns="torch/*", local_dir="/content/laya-release")) / "torch")
    SOURCE = f'{hub["repo"]}@{hub["revision"]}'
else:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT = "/content/laya-descent-v4"
    if not os.path.isdir(CHECKPOINT):
        zipfile.ZipFile("/content/drive/MyDrive/lowgate/laya-descent-v4.zip").extractall(CHECKPOINT)
    SOURCE = "Google Drive: lowgate/laya-descent-v4.zip"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 20):
            h.update(chunk)
    return h.hexdigest()

expected = {k[len("torch/"):]: v["sha256"] for k, v in manifest["files"].items() if k.startswith("torch/")}
wrong = [k for k, want in expected.items() if not os.path.isfile(f"{CHECKPOINT}/{k}") or sha256(f"{CHECKPOINT}/{k}") != want]
assert not wrong, f"These checkpoint files do not match the manifest: {wrong}"
print(f"{len(expected)} checkpoint files match the manifest, from {SOURCE}")


In [ ]:
# 5. Serve the PyTorch checkpoint through the same bridge the MLX build uses, listening on this Colab machine only.
import time, urllib.request
bridge = subprocess.Popen(
    ["python", "bridge/laya/server.py", "--backend", "torch", "--model", CHECKPOINT, "--device", "cuda"],
    cwd="/content/lowgate", stdout=open("/content/laya.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=5).read().decode())
        break
    except Exception:
        assert bridge.poll() is None, "The bridge stopped:\n" + open("/content/laya.log").read()[-3000:]
        time.sleep(5)
else:
    raise RuntimeError("The bridge did not start in 5 minutes:\n" + open("/content/laya.log").read()[-3000:])

In [ ]:
# 6. Ask the PyTorch build every recorded question. Prints one line per course, then the totals.
!pnpm -s agree --lane laya --courses {COURSES} --label {LABEL}

In [ ]:
# 7. Record what ran, then download the report. Unzip in the repo root on your computer.
import importlib.metadata as meta
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
json.dump({"commit": commit, "checkpoint": SOURCE, "laya": meta.version("laya"), "torch": meta.version("torch"), "node": NODE}, open(f"repro/laya-{LABEL}-env.json", "w"), indent=2)
!nvidia-smi --query-gpu=name --format=csv,noheader > repro/laya-{LABEL}-gpu.txt
!zip -qr /content/laya-repro.zip repro/laya-{LABEL}.json repro/laya-{LABEL}-gpu.txt repro/laya-{LABEL}-env.json
from google.colab import files
files.download("/content/laya-repro.zip")